***Read CSV Files***

In [0]:
path ="/Volumes/dev_account/stage/employee/Inbound"

df_employees = spark.read.format("csv").option("header","true").option("inferSchema","true").load(path).filter("is_active = true")


***Audit Fields***

In [0]:
from pyspark.sql.functions import lit,current_timestamp,current_user

df_audit=df_employees.withColumn("CreatedBy",lit(current_user()))\
    .withColumn("CreatedDate",current_timestamp())\
    .withColumn("UpdatedBy",lit(current_user()))\
    .withColumn("UpdatedDate",current_timestamp())\
    .withColumn("Start_Date",current_timestamp())\
    .withColumn("End_Date",lit('9999-12-12').cast("timestamp"))  # SCD2: NULL for active records

In [0]:
from pyspark.sql.functions import col

df_dim_account_schema=spark.table("dev_account.bronze.Dim_employees_Scd2").schema

df_select=df_audit.select(
    [col(field.name).cast(field.dataType) for field in df_dim_account_schema if field.name != "EmployeeSK"]
)
    

from delta.tables import *

In [0]:
display(df_select)